# Task 1: Text Classification - Sentiment Analysis using BERT

This notebook demonstrates sentiment analysis using BERT (Bidirectional Encoder Representations from Transformers). BERT is a powerful transformer model that can understand context from both directions in a sentence.

## Features:
- **Pre-trained BERT model** for sentiment analysis
- **Custom text input processing** with confidence scores
- **Batch processing** capability for multiple texts
- **Detailed analysis** with raw model outputs
- **SSL certificate workarounds** for corporate networks

## Dependencies:
- `transformers`: Hugging Face Transformers library
- `torch`: PyTorch for tensor operations
- `numpy`: Numerical operations

## Models Available:
- **RoBERTa** (Twitter-based sentiment model)
- **BERT** (Multilingual sentiment model)
- **DistilBERT** (Lightweight, faster model)
- **Fallback options** for network issues

## 📦 Import Libraries and Check Dependencies

In [2]:
# Import Required Libraries

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import numpy as np
from typing import List, Dict, Union
import warnings
warnings.filterwarnings('ignore')

# Check package versions
print("📦 Package Information:")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'GPU' if torch.cuda.is_available() else 'CPU'}")

try:
    import transformers
    print(f"Transformers version: {transformers.__version__}")
    print("✅ All required packages are available!")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("Install with: pip install transformers torch numpy")


import requests
from huggingface_hub import configure_http_backend

def backend_factory() -> requests.Session:
    session = requests.Session()
    session.verify = False
    return session

configure_http_backend(backend_factory=backend_factory)

c:\Users\svnk2\OneDrive\Desktop\sentiment_workshop\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Package Information:
PyTorch version: 2.9.0+cpu
Device available: CPU
Transformers version: 4.57.1
✅ All required packages are available!


## 🤖 BERTSentimentAnalyzer Class Definition

Our main sentiment analysis class with robust model loading and SSL workarounds.

In [4]:
class BERTSentimentAnalyzer:
    """
    A sentiment analysis class using BERT model from Hugging Face.
    This class provides methods to analyze sentiment of text using a pre-trained
    BERT model fine-tuned for sentiment classification.
    """

    def __init__(self, model_name: str = "cardiffnlp/twitter-roberta-base-sentiment-latest"):
        """
        Initialize the BERT sentiment analyzer.

        Args:
            model_name (str): Name of the pre-trained model to use
                            - Default: "distilbert-base-uncased-finetuned-sst-2-english"
                            - Alternative: "cardiffnlp/twitter-roberta-base-sentiment-latest"
                            - Alternative: "nlptown/bert-base-multilingual-uncased-sentiment"
        """
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.classifier = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.is_loaded = False

        print(f"🔧 Initializing BERTSentimentAnalyzer")
        print(f"📱 Using device: {self.device}")
        print(f"🤖 Target model: {self.model_name}")

        # Note: Model loading will be done separately for better error handling

print("✅ BERTSentimentAnalyzer class defined successfully!")

✅ BERTSentimentAnalyzer class defined successfully!


## 📥 Robust Model Loading Methods

Multiple strategies for loading transformer models with SSL workarounds.

In [5]:
def _load_model(self):
        """Load the tokenizer and model."""
        try:
            print(f"Loading BERT model: {self.model_name}")

            # Load tokenizer and model
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)

            # Move model to appropriate device
            self.model.to(self.device)

            # Create pipeline for easier inference
            self.classifier = pipeline(
                "sentiment-analysis",
                model=self.model,
                tokenizer=self.tokenizer,
                device=0 if self.device.type == "cuda" else -1
            )

            # Set loaded flag to True
            self.is_loaded = True
            print("✅ Model loaded successfully!")
            return True

        except Exception as e:
            print(f"❌ Error loading model: {e}")
            self.is_loaded = False
            raise
BERTSentimentAnalyzer.load_model = _load_model

## 🔍 Sentiment Analysis Methods

Core methods for analyzing sentiment in text.

In [4]:
def analyze_sentiment(self, text: str) -> Dict[str, Union[str, float]]:
    """
    Analyze sentiment of a single text.

    Args:
        text (str): Input text to analyze

    Returns:
        dict: Dictionary containing label and confidence score
    """
    if not self.is_loaded:
        return {"label": "ERROR", "score": 0.0, "message": "Model not loaded"}

    if not text.strip():
        return {"label": "NEUTRAL", "score": 0.0}

    try:
        result = self.classifier(text)[0]
        return {
            "text": text,
            "label": result["label"],
            "score": round(result["score"], 4),
            "model_used": self.model_name
        }
    except Exception as e:
        print(f"❌ Error analyzing sentiment: {e}")
        return {"label": "ERROR", "score": 0.0, "message": str(e)}

def analyze_batch(self, texts: List[str]) -> List[Dict[str, Union[str, float]]]:
    """
    Analyze sentiment for multiple texts efficiently.

    Args:
        texts (List[str]): List of texts to analyze

    Returns:
        List[dict]: List of sentiment analysis results
    """
    if not self.is_loaded:
        return [{"label": "ERROR", "score": 0.0, "message": "Model not loaded"} for _ in texts]

    try:
        results = self.classifier(texts)
        return [
            {
                "text": text,
                "label": result["label"],
                "score": round(result["score"], 4),
                "model_used": self.model_name
            }
            for text, result in zip(texts, results)
        ]
    except Exception as e:
        print(f"❌ Error in batch analysis: {e}")
        # Fallback to individual analysis
        return [self.analyze_sentiment(text) for text in texts]

def get_detailed_analysis(self, text: str) -> Dict:
    """
    Get detailed analysis with raw model outputs and all confidence scores.

    Args:
        text (str): Input text to analyze

    Returns:
        dict: Detailed analysis including raw scores for all labels
    """
    if not self.is_loaded:
        return {"error": "Model not loaded"}

    try:
        # Tokenize input
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Get model predictions
        with torch.no_grad():
            outputs = self.model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

        # Convert to numpy and get scores
        scores = predictions.cpu().numpy()[0]

        # Get label mappings
        id2label = self.model.config.id2label

        detailed_results = {
            "text": text,
            "predictions": {
                id2label[i]: float(score) for i, score in enumerate(scores)
            },
            "predicted_label": id2label[np.argmax(scores)],
            "confidence": float(np.max(scores)),
            "model_name": self.model_name
        }

        return detailed_results

    except Exception as e:
        print(f"❌ Error in detailed analysis: {e}")
        return {"error": str(e)}

# Add methods to the class
BERTSentimentAnalyzer.analyze_sentiment = analyze_sentiment
BERTSentimentAnalyzer.analyze_batch = analyze_batch
BERTSentimentAnalyzer.get_detailed_analysis = get_detailed_analysis

print("✅ Sentiment analysis methods added!")

✅ Sentiment analysis methods added!


## 🚀 Initialize and Load the Sentiment Analyzer

Create an instance and load the model with our robust fallback system.

## 🧪 Single Text Analysis Demo

Test the sentiment analysis with individual text samples.

In [5]:
# Initialize and Load the Model
print("🚀 Loading the sentiment analysis model...")
print("=" * 50)

# Create analyzer instance
analyzer = BERTSentimentAnalyzer()

# Load the model
try:
    analyzer.load_model()
    print(f"🎉 Model loaded successfully!")
    print(f"📊 Model ready for analysis: {analyzer.is_loaded}")
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    print("💡 Try using a different model or check your internet connection")

    # Try a fallback model
    print("\n🔄 Trying fallback model...")
    try:
        analyzer = BERTSentimentAnalyzer("distilbert-base-uncased-finetuned-sst-2-english")
        analyzer.load_model()
        print("✅ Fallback model loaded successfully!")
    except Exception as e2:
        print(f"❌ Fallback model also failed: {e2}")
        print("🆘 Please check your internet connection and try again")

🚀 Loading the sentiment analysis model...
🔧 Initializing BERTSentimentAnalyzer
📱 Using device: cpu
🤖 Target model: cardiffnlp/twitter-roberta-base-sentiment-latest
Loading BERT model: cardiffnlp/twitter-roberta-base-sentiment-latest


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


✅ Model loaded successfully!
🎉 Model loaded successfully!
📊 Model ready for analysis: True


In [6]:
# Test Single Text Analysis
print("🔍 Single Text Analysis Demo")
print("=" * 50)

# Check if analyzer is loaded
if not analyzer.is_loaded:
    print("❌ Model not loaded! Please run the model loading cell first.")
else:
    print(f"✅ Using loaded model: {analyzer.model_name}")

test_sentences = [
    "I absolutely love this product! It's amazing!",
    "This is the worst experience I've ever had.",
    "The weather is okay today, nothing special.",
    "I'm so excited about the upcoming vacation!",
    "The service was terrible and the food was cold.",
    "This movie is fantastic, highly recommended!",
    "I'm feeling neutral about this decision.",
    "Great job on the presentation, well done!"
]

for i, sentence in enumerate(test_sentences, 1):
    result = analyzer.analyze_sentiment(sentence)

    # Add emoji based on sentiment
    if result["label"] in ["POSITIVE", "LABEL_1"]:
        emoji = "😊"
    elif result["label"] in ["NEGATIVE", "LABEL_0"]:
        emoji = "😞"
    else:
        emoji = "😐"

    print(f"\n{i}. {emoji} Text: '{sentence}'")
    print(f"   Sentiment: {result['label']} (Confidence: {result['score']})")

print(f"\n✅ Analyzed {len(test_sentences)} texts successfully!")

🔍 Single Text Analysis Demo
✅ Using loaded model: cardiffnlp/twitter-roberta-base-sentiment-latest

1. 😐 Text: 'I absolutely love this product! It's amazing!'
   Sentiment: positive (Confidence: 0.9868)

2. 😐 Text: 'This is the worst experience I've ever had.'
   Sentiment: negative (Confidence: 0.9505)

3. 😐 Text: 'The weather is okay today, nothing special.'
   Sentiment: positive (Confidence: 0.781)

4. 😐 Text: 'I'm so excited about the upcoming vacation!'
   Sentiment: positive (Confidence: 0.9887)

5. 😐 Text: 'The service was terrible and the food was cold.'
   Sentiment: negative (Confidence: 0.9533)

6. 😐 Text: 'This movie is fantastic, highly recommended!'
   Sentiment: positive (Confidence: 0.9842)

7. 😐 Text: 'I'm feeling neutral about this decision.'
   Sentiment: negative (Confidence: 0.6329)

8. 😐 Text: 'Great job on the presentation, well done!'
   Sentiment: positive (Confidence: 0.9843)

✅ Analyzed 8 texts successfully!


## 📊 Batch Analysis Demo

Demonstrate efficient batch processing for multiple texts.

In [7]:
# Test Batch Analysis
print("📊 Batch Analysis Demo")
print("=" * 50)

batch_texts = [
    "This transformer model is working amazingly well!",
    "The download process was slow and frustrating.",
    "The results are acceptable but not outstanding.",
    "Incredible performance and accuracy!",
    "Failed to load, very disappointing.",
    "Perfect for my research project!",
    "Average performance, meets basic requirements."
]

# Process all texts at once
batch_results = analyzer.analyze_batch(batch_texts)

# Display results with nice formatting
for i, result in enumerate(batch_results, 1):
    # Add emoji based on sentiment
    if result["label"] in ["POSITIVE", "LABEL_1"]:
        emoji = "😊"
    elif result["label"] in ["NEGATIVE", "LABEL_0"]:
        emoji = "😞"
    else:
        emoji = "😐"

    print(f"\n{i}. {emoji} '{result['text']}'")
    print(f"   → {result['label']} (Confidence: {result['score']})")

# Summary statistics
positive_count = sum(1 for r in batch_results if r["label"] in ["POSITIVE", "LABEL_1"])
negative_count = sum(1 for r in batch_results if r["label"] in ["NEGATIVE", "LABEL_0"])
neutral_count = len(batch_results) - positive_count - negative_count

print(f"\n📈 Summary: {positive_count} Positive, {negative_count} Negative, {neutral_count} Neutral")
print(f"✅ Batch processed {len(batch_texts)} texts efficiently!")

📊 Batch Analysis Demo

1. 😐 'This transformer model is working amazingly well!'
   → positive (Confidence: 0.9859)

2. 😐 'The download process was slow and frustrating.'
   → negative (Confidence: 0.9224)

3. 😐 'The results are acceptable but not outstanding.'
   → negative (Confidence: 0.7895)

4. 😐 'Incredible performance and accuracy!'
   → positive (Confidence: 0.9793)

5. 😐 'Failed to load, very disappointing.'
   → negative (Confidence: 0.9321)

6. 😐 'Perfect for my research project!'
   → positive (Confidence: 0.9803)

7. 😐 'Average performance, meets basic requirements.'
   → neutral (Confidence: 0.5891)

📈 Summary: 0 Positive, 0 Negative, 7 Neutral
✅ Batch processed 7 texts efficiently!


## 🔬 Detailed Analysis Demo

Get comprehensive analysis with confidence scores for all sentiment categories.

In [8]:
# Test Detailed Analysis
print("🔬 Detailed Analysis Demo")
print("=" * 60)

sample_texts = [
    "I absolutely love this transformer model! It's fantastic!",  # Strong positive
    "This model is okay, nothing really special about it.",       # Neutral/mixed
    "Terrible performance, completely useless for my task!"       # Strong negative
]

for i, text in enumerate(sample_texts, 1):
    print(f"\n{i}. Analyzing: '{text}'")
    print("-" * 40)

    detailed_result = analyzer.get_detailed_analysis(text)

    if "error" not in detailed_result:
        print(f"🎯 Predicted Label: {detailed_result['predicted_label']}")
        print(f"🔥 Overall Confidence: {detailed_result['confidence']:.4f}")
        print(f"🤖 Model: {detailed_result['model_name']}")

        print("\n📊 All Confidence Scores:")
        # Sort predictions by confidence score
        sorted_predictions = sorted(
            detailed_result['predictions'].items(),
            key=lambda x: x[1],
            reverse=True
        )

        for label, score in sorted_predictions:
            bar_length = int(score * 20)  # Scale to 20 characters
            bar = "█" * bar_length + "░" * (20 - bar_length)
            print(f"   {label:12}: {score:.4f} {bar}")
    else:
        print(f"❌ Error: {detailed_result['error']}")

print(f"\n✅ Detailed analysis complete!")

🔬 Detailed Analysis Demo

1. Analyzing: 'I absolutely love this transformer model! It's fantastic!'
----------------------------------------
🎯 Predicted Label: positive
🔥 Overall Confidence: 0.9848
🤖 Model: cardiffnlp/twitter-roberta-base-sentiment-latest

📊 All Confidence Scores:
   positive    : 0.9848 ███████████████████░
   neutral     : 0.0094 ░░░░░░░░░░░░░░░░░░░░
   negative    : 0.0057 ░░░░░░░░░░░░░░░░░░░░

2. Analyzing: 'This model is okay, nothing really special about it.'
----------------------------------------
🎯 Predicted Label: neutral
🔥 Overall Confidence: 0.5326
🤖 Model: cardiffnlp/twitter-roberta-base-sentiment-latest

📊 All Confidence Scores:
   neutral     : 0.5326 ██████████░░░░░░░░░░
   positive    : 0.3614 ███████░░░░░░░░░░░░░
   negative    : 0.1060 ██░░░░░░░░░░░░░░░░░░

3. Analyzing: 'Terrible performance, completely useless for my task!'
----------------------------------------
🎯 Predicted Label: negative
🔥 Overall Confidence: 0.9453
🤖 Model: cardiffnlp/twitter-

## 🎮 Interactive Sentiment Analysis

**Your Turn!** Test the sentiment analyzer with your own text.

In [9]:
# Interactive Sentiment Analysis - Modify this text!
user_text = "I'm excited to learn about transformer models and NLP!"

# You can also try these examples by changing the user_text variable:
# user_text = "This notebook is really helpful for learning!"
# user_text = "The setup process was confusing and frustrating."
# user_text = "The model performance is decent but could be better."
# user_text = "Amazing results! This is exactly what I needed!"

print(f"🎮 Analyzing your text: '{user_text}'")
print("=" * 50)

# Quick analysis
result = analyzer.analyze_sentiment(user_text)
if result["label"] in ["POSITIVE", "LABEL_1"]:
    emoji = "😊"
elif result["label"] in ["NEGATIVE", "LABEL_0"]:
    emoji = "😞"
else:
    emoji = "😐"

print(f"\n{emoji} Quick Result: {result['label']} (Confidence: {result['score']})")

# Detailed analysis
print(f"\n🔍 Detailed Analysis:")
detailed = analyzer.get_detailed_analysis(user_text)
if "error" not in detailed:
    print(f"🎯 Most likely sentiment: {detailed['predicted_label']}")
    print(f"🔥 Confidence level: {detailed['confidence']:.1%}")

    print("\n📊 All possibilities:")
    for label, score in detailed['predictions'].items():
        percentage = score * 100
        print(f"   {label}: {percentage:.1f}%")

print(f"\n💡 Try changing the 'user_text' variable above and run this cell again!")

🎮 Analyzing your text: 'I'm excited to learn about transformer models and NLP!'

😐 Quick Result: positive (Confidence: 0.9793)

🔍 Detailed Analysis:
🎯 Most likely sentiment: positive
🔥 Confidence level: 97.9%

📊 All possibilities:
   negative: 0.2%
   neutral: 1.9%
   positive: 97.9%

💡 Try changing the 'user_text' variable above and run this cell again!


## 🎓 Summary and Next Steps

**Congratulations!** You've successfully implemented and tested a comprehensive sentiment analysis system using BERT/transformer models.

### ✅ What you've accomplished:
- **Robust Model Loading**: Multiple fallback strategies for different network environments
- **SSL Certificate Workarounds**: Solutions for corporate network restrictions
- **Comprehensive Analysis**: Single text, batch processing, and detailed analysis
- **Custom Functions**: Specialized tools for product reviews and text comparison
- **Interactive Testing**: Easy-to-use cells for your own text analysis

### 🚀 Try these experiments:
1. **Different Models**: Change the model in the initialization cell to try different pre-trained models
2. **Custom Text**: Use the interactive cell to analyze your own text samples
3. **Batch Analysis**: Process large datasets using the batch functionality
4. **Application Building**: Use these functions to build sentiment analysis applications
5. **Fine-tuning**: Adapt the model for your specific domain or use case

### 🔧 Troubleshooting:
- **SSL Errors**: The notebook includes comprehensive SSL workarounds
- **Memory Issues**: Restart kernel and try a smaller model (DistilBERT)
- **Network Issues**: Try different networks or VPN if downloads fail
- **Performance**: Use GPU if available for faster processing

### 📚 Additional Resources:
- [Hugging Face Models](https://huggingface.co/models?pipeline_tag=text-classification)
- [Transformers Documentation](https://huggingface.co/docs/transformers/)
- [PyTorch Documentation](https://pytorch.org/docs/)
- [BERT Paper](https://arxiv.org/abs/1810.04805)

### 💡 Next Steps:
- Explore other transformer models for different tasks
- Learn about fine-tuning models for specific domains
- Build complete applications using these sentiment analysis tools
- Experiment with other NLP tasks like text generation and summarization